<a href="https://colab.research.google.com/github/Gransasso67/option-screener/blob/main/put_screener_v22_qualita_contesto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Screener put coperte — Versione 2.2: qualità + contesto di prezzo

Due assi separati:
1. **Qualità** (bilanci SEC): solidità difensiva e cash flow, come nella versione 1, più il **trend dei fondamentali** (ultimo anno / TTM rispetto alla media).
2. **Contesto di prezzo**: drawdown, trend, volatilità, valutazione (FCF yield, P/E rispetto alla propria storia).

**Novità della 2.2:** alternativa per il cash flow operativo da attività continuative (caso DOV), capex alternativi estesi (software capitalizzato), ultimo esercizio senza FCF = *Dati incompleti*, nuovo stato *Candidato con cautela* per le bandiere gialle, report della copertura dei dati.

**Novità della 2.1:** TTM che non usa più dati vecchi, capex alternativi quando il tag standard manca (caso TPL), indicatori "peggior anno" di FCF ed EBIT,
trend confrontato sia con l'ultimo esercizio sia con la mediana, causa del calo scomposta in utili / cassa / multiplo (variazioni a 12 mesi),
bandiere di prezzo con persistenza (meno sfarfallio), stato *Dati incompleti*, termine di confronto per la correlazione.

Poi li combina in una matrice (*Candidato* / *Da osservare* / *Fuori*), limita i titoli per sottosettore, misura quanto sono correlati tra loro
e permette di verificare i **casi di controllo** (titoli già calati) mese per mese.

**Prima di eseguire:** inserisci nome ed email reali in `USER_AGENT` (richiesto dalla SEC). Esegui le celle in ordine.
La prima esecuzione richiede alcuni minuti (bilanci + prezzi); poi i bilanci restano in cache.

In [ ]:
import io, json, time, requests
import numpy as np
import pandas as pd
from pathlib import Path
try:
    import yfinance as yf
except ImportError:
    get_ipython().system("pip -q install yfinance")
    import yfinance as yf

# ---------------- CONFIGURAZIONE ----------------
USER_AGENT = "Nome Cognome nome.cognome@esempio.it"   # <-- modificare: la SEC richiede nome + email
CACHE_DIR = Path("cache_put_screener_v21"); CACHE_DIR.mkdir(exist_ok=True)   # cartella nuova: dati con TTM e capex corretti
CACHE_MAX_GIORNI = 30
N_ANNI = 5
ESCLUDI_SETTORI = {"Financials", "Real Estate"}
CONFRONTO = "settore"          # percentili di qualità: "settore" o "universo"
DATO_MANCANTE = 0.25           # percentile assegnato quando un dato manca

# ---- Asse 1: qualità
PESI = {"Solidità patrimoniale": 0.30, "Cash flow": 0.30, "Stabilità utili": 0.15,
        "Redditività": 0.10, "Remunerazione azionisti": 0.15}
SOGLIE = dict(fcf_pos_min=0.8, utile_pos_min=0.6, int_cov_min=2.0, nd_ebitda_max=6.0)
SOGLIA_TREND_EBIT = -0.25      # bandiera se l'EBIT (TTM) è sotto l'ultimo esercizio O la mediana degli ultimi esercizi di oltre il 25%
SOGLIA_TREND_FCF = -0.35       # idem per il FCF, soglia più larga perché più volatile

# ---- Asse 2: contesto di prezzo
PESI_CTX = {"Trend e momentum": 0.40, "Rischio storico": 0.25, "Valutazione": 0.35}
DD_ROSSO = 0.35                # drawdown da massimo 52 settimane che, con prezzo sotto media 200gg, accende la bandiera rossa
DD_GIALLO = 0.20
PERSISTENZA = 0.60             # una bandiera di prezzo si accende se la condizione vale per almeno il 60% degli ultimi 20 giorni di borsa

# ---- Combinazione e selezione
QUALITA_PERCENTILE = 0.70      # "qualità alta" = primo 30% dei titoli ammessi
MAX_PER_SOTTOSETTORE = 2
CORR_ALTA = 0.70               # correlazione oltre cui due titoli contano come "lo stesso rischio"
CASI_CONTROLLO = ["NRG", "TPL", "GOOG", "CDNS", "VST", "AOS"]
CASI_SANI = []                 # esempi: ["JNJ", "PG", "KO"], titoli sani per misurare i falsi allarmi

## 1. Universo: componenti S&P 500 (settore, sottosettore, CIK)

In [ ]:
def carica_universo():
    r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                     headers={"User-Agent": USER_AGENT}, timeout=30)
    r.raise_for_status()
    df = pd.read_html(io.StringIO(r.text))[0]
    df = df.rename(columns={"Symbol": "ticker", "Security": "nome", "GICS Sector": "settore",
                            "GICS Sub-Industry": "sottosettore", "CIK": "cik"})
    df["cik"] = df["cik"].astype(int).astype(str).str.zfill(10)
    # una società con più classi di azioni (GOOGL/GOOG, FOXA/FOX, NWSA/NWS) conta una volta sola
    df = df.drop_duplicates("cik", keep="last")
    return df[["ticker", "nome", "settore", "sottosettore", "cik"]]

## 2. Bilanci da EDGAR: annuali, TTM, azioni in circolazione (con cache)

In [ ]:
import re
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"})

TAGS = {
    "ricavi":    (["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax",
                   "RevenueFromContractWithCustomerIncludingAssessedTax", "SalesRevenueNet"], True),
    "ebit":      (["OperatingIncomeLoss"], True),
    "utile":     (["NetIncomeLoss", "ProfitLoss", "NetIncomeLossAvailableToCommonStockholdersBasic"], True),
    "cfo":       (["NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"], True),
    "capex":     (["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets"], True),
    "ammort":    (["DepreciationDepletionAndAmortization", "DepreciationAndAmortization",
                   "DepreciationAmortizationAndAccretionNet"], True),
    "interessi": (["InterestExpense", "InterestExpenseDebt", "InterestExpenseNonoperating"], True),
    "dividendi": (["PaymentsOfDividends", "PaymentsOfDividendsCommonStock"], True),
    "buyback":   (["PaymentsForRepurchaseOfCommonStock"], True),
    "azioni":    (["WeightedAverageNumberOfDilutedSharesOutstanding"], True),
    "cassa":     (["CashAndCashEquivalentsAtCarryingValue",
                   "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"], False),
    "debito_lt": (["LongTermDebtNoncurrent", "LongTermDebt"], False),
    "debito_bt": (["LongTermDebtCurrent", "DebtCurrent"], False),
    "attivo":    (["Assets"], False),
    "att_corr":  (["AssetsCurrent"], False),
    "pass_corr": (["LiabilitiesCurrent"], False),
}
# Tag alternativi per gli investimenti, usati solo se quelli standard non coprono gli ultimi esercizi
CAPEX_PAROLE = ("PropertyPlant", "ProductiveAssets", "OilAndGas", "RealEstate", "Land", "CapitalImprovements", "Mineral", "Royalty", "Software")
CAPEX_ESCLUDI = ("Business", "Investment", "Intangible", "Securities")

def _serie(facts, tag, flow):
    node = facts.get("us-gaap", {}).get(tag)
    if not node:
        return pd.Series(dtype=float)
    unit = "shares" if tag.startswith("WeightedAverage") else "USD"
    rows = node.get("units", {}).get(unit, [])
    if not rows:
        return pd.Series(dtype=float)
    df = pd.DataFrame(rows)
    df = df[df["form"].isin(["10-K", "10-K/A"])].copy()
    if df.empty:
        return pd.Series(dtype=float)
    df["end"] = pd.to_datetime(df["end"])
    if flow:
        if "start" not in df.columns:
            return pd.Series(dtype=float)
        df["start"] = pd.to_datetime(df["start"])
        durata = (df["end"] - df["start"]).dt.days
        df = df[(durata >= 350) & (durata <= 380)]
    df = df.sort_values("filed").drop_duplicates("end", keep="last")
    return df.set_index("end")["val"].astype(float)

def _per_anno(s):
    """Da indice data a indice anno fiscale (fine esercizio - 45 giorni). Ritorna (serie, {anno: data di chiusura})."""
    if s.empty:
        return s, {}
    etich = (s.index - pd.Timedelta(days=45)).year
    mappa = dict(zip(etich, s.index))
    s2 = s.copy()
    s2.index = etich
    return s2[~s2.index.duplicated(keep="last")].sort_index(), mappa

def _tag_capex_alternativo(facts, anni_mancanti):
    if not anni_mancanti:
        return None
    best, best_score = None, (0, 0.0)
    for tag in facts.get("us-gaap", {}):
        if not tag.startswith(("PaymentsToAcquire", "PaymentsToDevelop", "PaymentsForCapitalImprovements", "PaymentsForSoftware")):
            continue
        if not any(p in tag for p in CAPEX_PAROLE) or any(x in tag for x in CAPEX_ESCLUDI) or tag in TAGS["capex"][0]:
            continue
        s, _ = _per_anno(_serie(facts, tag, True))
        cop = len(set(s.index) & anni_mancanti)
        if cop and (cop, float(s.abs().sum())) > best_score:
            best, best_score = tag, (cop, float(s.abs().sum()))
    return best

def estrai_serie(facts):
    """Tabella anni x voci. Colonne extra: fine_es (data di chiusura) e capex_fonte (standard / alt:<tag> / mancante)."""
    out, fine = {}, {}
    for nome, (tags, flow) in TAGS.items():
        s = pd.Series(dtype=float)
        for t in tags:
            s = s.combine_first(_serie(facts, t, flow))
        s, mappa = _per_anno(s)
        if nome in ("ricavi", "ebit", "utile", "cfo"):
            for e, d in mappa.items():
                fine[e] = max(fine.get(e, d), d)
        out[nome] = s
    # capex: se mancano esercizi rispetto al cash flow operativo, prova un tag alternativo
    anni_cfo = set(out["cfo"].dropna().index[-N_ANNI:])
    mancanti = anni_cfo - set(out["capex"].dropna().index)
    alt = _tag_capex_alternativo(facts, mancanti)
    fonte = "standard" if out["capex"].notna().any() else "mancante"
    if alt:
        s_alt, _ = _per_anno(_serie(facts, alt, True))
        out["capex"] = out["capex"].combine_first(s_alt)
        fonte = "alt:" + alt
    df = pd.DataFrame(out)
    df["fine_es"] = pd.Series({k: v.strftime("%Y-%m-%d") for k, v in fine.items()})
    df["capex_fonte"] = fonte
    df["capex_mancanti"] = len(anni_cfo - set(out["capex"].dropna().index))     # esercizi con cash flow ma senza capex
    return df

# ---- TTM: ultimo esercizio + parziale corrente - parziale dello stesso periodo dell'anno prima
def _fatti_flusso(facts, tags):
    """Tutti i valori di flusso (10-K e 10-Q, qualsiasi durata) per (inizio, fine); i primi tag hanno priorità."""
    out = {}
    for i, t in enumerate(tags):
        node = facts.get("us-gaap", {}).get(t)
        if not node:
            continue
        for r in node.get("units", {}).get("USD", []):
            if "start" not in r or r.get("form") not in ("10-K", "10-K/A", "10-Q", "10-Q/A"):
                continue
            k = (r["start"], r["end"])
            if k not in out or (out[k][2] == i and r["filed"] > out[k][0]):
                out[k] = (r["filed"], float(r["val"]), i)
    return out

def _ttm(fatti, rif=None):
    """(valore TTM, data di fine del periodo) alla data rif (default: ultimo dato disponibile)."""
    if not fatti:
        return np.nan, None
    df = pd.DataFrame([(pd.Timestamp(s), pd.Timestamp(e), v) for (s, e), (f, v, i) in fatti.items()],
                      columns=["start", "end", "val"])
    if rif is not None:
        df = df[df["end"] <= rif]
    df["dur"] = (df["end"] - df["start"]).dt.days
    ann = df[(df["dur"] >= 350) & (df["dur"] <= 380)].sort_values("end")
    if ann.empty:
        return np.nan, None
    fy = ann.iloc[-1]
    cand = df[((df["start"] - fy["end"]).dt.days.abs() <= 7) & (df["dur"] >= 70) & (df["dur"] <= 290)]
    if cand.empty:
        return float(fy["val"]), fy["end"]               # nessun trimestre più recente dell'ultimo 10-K
    cur = cand.sort_values("end").iloc[-1]
    prec = df[((df["start"] - (cur["start"] - pd.Timedelta(days=365))).dt.days.abs() <= 10) &
              ((df["end"] - (cur["end"] - pd.Timedelta(days=365))).dt.days.abs() <= 10)]
    if prec.empty:
        return np.nan, None
    return float(fy["val"] + cur["val"] - prec.iloc[0]["val"]), cur["end"]

def _azioni_ultime(facts, rif=None):
    node = facts.get("us-gaap", {}).get("WeightedAverageNumberOfDilutedSharesOutstanding")
    if not node:
        return np.nan
    rows = [r for r in node.get("units", {}).get("shares", [])
            if r.get("form") in ("10-K", "10-K/A", "10-Q", "10-Q/A") and (rif is None or pd.Timestamp(r["end"]) <= rif)]
    if not rows:
        return np.nan
    return float(max(rows, key=lambda x: (x["end"], x["filed"]))["val"])

def calcola_ttm(facts, capex_alt=None, rif=None):
    """TTM di ricavi, EBIT, utile, cash flow operativo, capex. Un TTM il cui ultimo periodo è più vecchio di
    60 giorni rispetto a quello di EBIT/ricavi/cash flow viene scartato (niente dati vecchi mescolati a dati nuovi)."""
    ris = {}
    for n in ("ricavi", "ebit", "utile", "cfo", "capex"):
        tags = TAGS[n][0] + ([capex_alt] if n == "capex" and capex_alt else [])
        ris[n] = _ttm(_fatti_flusso(facts, tags), rif)
    fini = [ris[n][1] for n in ("ricavi", "ebit", "cfo") if ris[n][1] is not None]
    fine_max = max(fini) if fini else None
    out = {}
    for n, (v, d) in ris.items():
        valido = d is not None and fine_max is not None and (fine_max - d).days <= 60
        out[n + "_ttm"] = v if valido else np.nan
    out["fcf_ttm"] = out["cfo_ttm"] - abs(out["capex_ttm"])
    out["azioni_ult"] = _azioni_ultime(facts, rif)
    out["fine_ttm"] = fine_max.strftime("%Y-%m-%d") if fine_max is not None else None
    return out

def _ser(v):
    if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
        return None
    return v if isinstance(v, str) else float(v)

def _de_ser(k, v):
    return v if k == "fine_ttm" else (np.nan if v is None else float(v))

def scarica(cik, ticker):
    f_a, f_t = CACHE_DIR / f"{ticker}.csv", CACHE_DIR / f"{ticker}_ttm.json"
    ok = lambda f: f.exists() and (time.time() - f.stat().st_mtime) < CACHE_MAX_GIORNI * 86400
    if ok(f_a) and ok(f_t):
        with open(f_t) as fh:
            return pd.read_csv(f_a, index_col=0), {k: _de_ser(k, v) for k, v in json.load(fh).items()}
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    for tentativo in range(3):
        r = SESSION.get(url, timeout=30)
        if r.status_code == 200:
            break
        time.sleep(1 + tentativo)
    else:
        return None, None
    facts = r.json().get("facts", {})
    df = estrai_serie(facts)
    fonte = df["capex_fonte"].iloc[0] if len(df) else "mancante"
    alt = fonte[4:] if fonte.startswith("alt:") else None
    ttm = calcola_ttm(facts, alt)
    if ttm["fine_ttm"]:                       # utile e azioni di un anno prima, per scomporre il calo del prezzo
        t1 = calcola_ttm(facts, alt, rif=pd.Timestamp(ttm["fine_ttm"]) - pd.Timedelta(days=350))
        ttm["utile_ttm_1y"], ttm["azioni_1y"] = t1["utile_ttm"], t1["azioni_ult"]
    df.to_csv(f_a)
    with open(f_t, "w") as fh:
        json.dump({k: _ser(v) for k, v in ttm.items()}, fh)
    time.sleep(0.15)
    return df, ttm

## 3. Indicatori di qualità e trend dei fondamentali

In [ ]:
def _trend(att, riferimenti):
    """Variazione di 'att' rispetto a ciascun riferimento positivo; restituisce la peggiore."""
    v = [att / r - 1 for r in riferimenti if pd.notna(r) and r > 0 and pd.notna(att)]
    return min(v) if v else np.nan

def metriche(df, ttm):
    d = df.drop(columns=["fine_es", "capex_fonte", "capex_mancanti"], errors="ignore").sort_index().tail(N_ANNI)
    if len(d) < 3 or d["ricavi"].notna().sum() < 3:
        return None
    l = d.iloc[-1]
    debito = d["debito_lt"].fillna(0) + d["debito_bt"].fillna(0)
    ebitda = d["ebit"] + d["ammort"].fillna(0)
    fcf = d["cfo"] - d["capex"].abs()
    m = {"anni_dati": len(d), "capex_fonte": df["capex_fonte"].iloc[0] if "capex_fonte" in df else "n.d.",
         "capex_mancanti": int(df["capex_mancanti"].iloc[0]) if "capex_mancanti" in df else 0}

    # --- Solidità patrimoniale
    nd = debito.iloc[-1] - (l["cassa"] if pd.notna(l["cassa"]) else 0)
    m["nd_ebitda"] = max(nd / ebitda.iloc[-1], 0) if ebitda.iloc[-1] > 0 else np.nan
    if debito.iloc[-1] == 0:
        m["int_cov"] = 50.0
    else:
        ic = d["ebit"].tail(3).mean() / d["interessi"].abs().tail(3).mean()
        m["int_cov"] = float(np.clip(ic, 0, 50)) if np.isfinite(ic) else np.nan
    cr = l["att_corr"] / l["pass_corr"] if pd.notna(l["pass_corr"]) and l["pass_corr"] > 0 else np.nan
    m["curr_ratio"] = float(np.clip(cr, 0, 4)) if np.isfinite(cr) else np.nan

    # --- Cash flow (incluso il peggior anno: la solidità si vede negli anni difficili)
    fm = fcf / d["ricavi"]
    m["fcf_margin"] = fm.mean()
    m["fcf_min_margin"] = fm.min() if fm.notna().any() else np.nan
    m["fcf_conv"] = (fcf / d["utile"]).where(d["utile"] > 0).clip(0, 3).mean()
    f = fcf.dropna()
    m["fcf_pos"] = (f > 0).mean() if len(f) else np.nan

    # --- Stabilità e redditività
    marg = d["ebit"] / d["ricavi"]
    m["std_margine"] = marg.std()
    m["ebit_min_margin"] = marg.min() if marg.notna().any() else np.nan
    m["op_margin"] = marg.mean()
    m["utile_pos"] = (d["utile"].dropna() > 0).mean() if d["utile"].notna().any() else np.nan
    roce = d["ebit"] / (d["attivo"] - d["pass_corr"])
    m["roce"] = float(np.clip(roce.tail(3).mean(), -0.5, 0.6)) if roce.notna().any() else np.nan

    # --- Remunerazione degli azionisti
    div, bb = d["dividendi"].abs().fillna(0), d["buyback"].abs().fillna(0)
    fcf_tot = fcf.sum(min_count=1)
    m["payout_fcf"] = float(np.clip((div.sum() + bb.sum()) / fcf_tot, 0, 1)) if pd.notna(fcf_tot) and fcf_tot > 0 else 0.0
    m["div_years"] = (div > 0).mean()
    az = d["azioni"].dropna()
    m["sh_cagr"] = (az.iloc[-1] / az.iloc[0]) ** (1 / (len(az) - 1)) - 1 if len(az) >= 3 and az.iloc[0] > 0 else np.nan

    # --- Trend dei fondamentali: valore attuale (TTM se disponibile, altrimenti ultimo anno)
    #     confrontato con l'ultimo esercizio E con la mediana degli esercizi: conta il confronto peggiore
    ebit_att = ttm.get("ebit_ttm", np.nan)
    ebit_att = ebit_att if np.isfinite(ebit_att) else l["ebit"]
    m["trend_ebit"] = _trend(ebit_att, [d["ebit"].iloc[-1], d["ebit"].median()])
    fcf_att = ttm.get("fcf_ttm", np.nan)
    fcf_att = fcf_att if np.isfinite(fcf_att) else (fcf.iloc[-1] if pd.notna(fcf.iloc[-1]) else np.nan)
    m["trend_fcf"] = _trend(fcf_att, [fcf.iloc[-1], fcf.median()])
    m["ttm_usato"] = bool(np.isfinite(ttm.get("ebit_ttm", np.nan)))
    m["fcf_att"] = fcf_att
    m["fcf_ultimo_nd"] = bool(pd.isna(fcf.iloc[-1]))        # ultimo esercizio senza FCF calcolabile

    # --- Segnalazioni (non entrano nel punteggio)
    m["div_oltre_fcf"] = bool(pd.notna(fcf_tot) and fcf_tot > 0 and div.sum() > fcf_tot)
    m["ebit_ultimo"] = l["ebit"]
    return m

## 4. Punteggio di qualità

In [ ]:
DEFINIZIONE = {
    "nd_ebitda":       ("Solidità patrimoniale", 0.40, False),
    "int_cov":         ("Solidità patrimoniale", 0.35, True),
    "curr_ratio":      ("Solidità patrimoniale", 0.25, True),
    "fcf_margin":      ("Cash flow", 0.30, True),
    "fcf_min_margin":  ("Cash flow", 0.30, True),
    "fcf_conv":        ("Cash flow", 0.20, True),
    "fcf_pos":         ("Cash flow", 0.20, True),
    "std_margine":     ("Stabilità utili", 0.40, False),
    "ebit_min_margin": ("Stabilità utili", 0.30, True),
    "utile_pos":       ("Stabilità utili", 0.30, True),
    "roce":            ("Redditività", 0.60, True),
    "op_margin":       ("Redditività", 0.40, True),
    "payout_fcf":      ("Remunerazione azionisti", 0.40, True),
    "div_years":       ("Remunerazione azionisti", 0.30, True),
    "sh_cagr":         ("Remunerazione azionisti", 0.30, False),
}

def motivo_esclusione(r):
    mot = []
    if pd.notna(r.fcf_pos) and r.fcf_pos < SOGLIE["fcf_pos_min"]: mot.append("FCF spesso negativo")
    if pd.notna(r.utile_pos) and r.utile_pos < SOGLIE["utile_pos_min"]: mot.append("perdite frequenti")
    if pd.notna(r.int_cov) and r.int_cov < SOGLIE["int_cov_min"]: mot.append("copertura interessi bassa")
    if pd.notna(r.nd_ebitda) and r.nd_ebitda > SOGLIE["nd_ebitda_max"]: mot.append("debito netto/EBITDA alto")
    if pd.notna(r.ebit_ultimo) and r.ebit_ultimo <= 0: mot.append("risultato operativo negativo")
    return "; ".join(mot)

def calcola_punteggi(tab):
    tab = tab.copy()
    tab["esclusione"] = tab.apply(motivo_esclusione, axis=1)
    gruppo = tab["settore"] if CONFRONTO == "settore" else pd.Series("tutti", index=tab.index)
    pct = pd.DataFrame(index=tab.index)
    for col, (_, _, alto_meglio) in DEFINIZIONE.items():
        pct[col] = tab[col].groupby(gruppo).rank(pct=True, ascending=alto_meglio)
    tab["dati_mancanti"] = pct.isna().sum(axis=1)
    pct = pct.fillna(DATO_MANCANTE)
    for pil in PESI:
        cols = [c for c, (p, _, _) in DEFINIZIONE.items() if p == pil]
        w = np.array([DEFINIZIONE[c][1] for c in cols])
        tab[pil] = (pct[cols] * w).sum(axis=1) / w.sum() * 100
    tab["PUNTEGGIO"] = sum(tab[p] * w for p, w in PESI.items())
    return tab.sort_values("PUNTEGGIO", ascending=False).reset_index(drop=True)

## 5. Contesto di prezzo: indicatori, valutazione, punteggio, bandiere e quadrante

In [ ]:
def scarica_prezzi(tickers, chunk=80):
    """Prezzi giornalieri corretti per split e dividendi, ultimi 5 anni (colonne = ticker Yahoo)."""
    parti = []
    for i in range(0, len(tickers), chunk):
        blocco = tickers[i:i + chunk]
        for tentativo in range(3):
            try:
                d = yf.download(blocco, period="5y", auto_adjust=True, progress=False, threads=True)
                c = d["Close"] if isinstance(d.columns, pd.MultiIndex) else d[["Close"]].rename(columns={"Close": blocco[0]})
                parti.append(c)
                break
            except Exception:
                time.sleep(2)
    px = pd.concat(parti, axis=1)
    px.index = pd.to_datetime(px.index).tz_localize(None)
    return px.loc[:, ~px.columns.duplicated()]

def serie_bandiere(s):
    """Indicatori giornalieri e bandiere di prezzo. La bandiera si accende solo se la condizione vale per almeno
    PERSISTENZA degli ultimi 20 giorni di borsa (meno sfarfallio); 'grezza' è la versione senza persistenza."""
    sma = s.rolling(200).mean()
    mx = s.rolling(252, min_periods=60).max()
    df = pd.DataFrame({"prezzo": s, "dd_52w": s / mx - 1, "vs_sma200": s / sma - 1, "pend_sma200": sma / sma.shift(21) - 1})
    sotto = df["vs_sma200"] < 0
    c_r = (df["dd_52w"] <= -DD_ROSSO) & sotto
    c_g = sotto & ((df["pend_sma200"] < 0) | (df["dd_52w"] <= -DD_GIALLO))
    sr = c_r.astype(float).rolling(20).mean()
    sg = c_g.astype(float).rolling(20).mean()
    nd = df["vs_sma200"].isna()
    df["bandiera"] = np.where(nd, "n.d.", np.where(sr >= PERSISTENZA, "ROSSA", np.where(sg >= PERSISTENZA, "GIALLA", "VERDE")))
    df["grezza"] = np.where(nd, "n.d.", np.where(c_r, "ROSSA", np.where(c_g, "GIALLA", "VERDE")))
    return df

def metriche_prezzo(s):
    s = s.dropna()
    if len(s) < 260:
        return None
    p = s.iloc[-1]
    sma = s.rolling(200).mean()
    ret = s.pct_change().dropna()
    return dict(
        prezzo=p,
        prezzo_1y=s.iloc[-253],
        var_prezzo_12m=p / s.iloc[-253] - 1,
        dd_52w=p / s.iloc[-252:].max() - 1,
        dd_3y=p / s.iloc[-756:].max() - 1,
        dist_sma200=p / sma.iloc[-1] - 1,
        pendenza_sma200=sma.iloc[-1] / sma.iloc[-21] - 1,
        mom_12_1=s.iloc[-22] / s.iloc[-253] - 1,            # rendimento a 12 mesi escluso l'ultimo mese
        vol_1y=ret.iloc[-252:].std() * np.sqrt(252),
        maxdd_5y=(s / s.cummax() - 1).min(),
        bandiera_prezzo=serie_bandiere(s)["bandiera"].iloc[-1],
    )

def pe_storia(df, serie_prezzo):
    """Mediana del P/E (prezzo a fine esercizio / utile per azione) sugli ultimi esercizi."""
    d = df.dropna(subset=["utile", "azioni", "fine_es"]).tail(N_ANNI + 1)
    vals = []
    for _, r in d.iterrows():
        if r["utile"] <= 0 or r["azioni"] <= 0:
            continue
        p = serie_prezzo.dropna().asof(pd.Timestamp(r["fine_es"]))
        if pd.notna(p):
            vals.append(p / (r["utile"] / r["azioni"]))
    return float(np.median(vals)) if len(vals) >= 3 else np.nan

def scomposizione_12m(tt, mp):
    """Variazione a 12 mesi di utile per azione (TTM) e di P/E: il calo del prezzo viene da utili o da multiplo?"""
    az, az1 = tt.get("azioni_ult", np.nan), tt.get("azioni_1y", np.nan)
    eps = tt.get("utile_ttm", np.nan) / az if np.isfinite(az) and az > 0 else np.nan
    eps1 = tt.get("utile_ttm_1y", np.nan) / az1 if np.isfinite(az1) and az1 > 0 else np.nan
    var_eps = eps / eps1 - 1 if np.isfinite(eps) and np.isfinite(eps1) and eps1 > 0 else np.nan
    pe = mp["prezzo"] / eps if np.isfinite(eps) and eps > 0 else np.nan
    pe1 = mp["prezzo_1y"] / eps1 if np.isfinite(eps1) and eps1 > 0 else np.nan
    var_pe = pe / pe1 - 1 if np.isfinite(pe) and np.isfinite(pe1) else np.nan
    return eps, pe, var_eps, var_pe

DEF_CTX = {   # metrica: (pilastro, peso interno, True se "più alto = meglio")
    "mom_12_1":    ("Trend e momentum", 0.35, True),
    "dd_52w":      ("Trend e momentum", 0.35, True),
    "dist_sma200": ("Trend e momentum", 0.30, True),
    "vol_1y":      ("Rischio storico", 0.50, False),
    "maxdd_5y":    ("Rischio storico", 0.50, True),
    "fcf_yield":   ("Valutazione", 0.50, True),
    "pe_rel":      ("Valutazione", 0.50, False),
}

def score_contesto(tab):
    tab = tab.copy()
    pct = pd.DataFrame(index=tab.index)
    for col, (_, _, alto) in DEF_CTX.items():
        pct[col] = tab[col].rank(pct=True, ascending=alto)
    tab["dati_mancanti_ctx"] = pct.isna().sum(axis=1)
    pct = pct.fillna(DATO_MANCANTE)
    for pil in PESI_CTX:
        cols = [c for c, (p, _, _) in DEF_CTX.items() if p == pil]
        w = np.array([DEF_CTX[c][1] for c in cols])
        tab[pil] = (pct[cols] * w).sum(axis=1) / w.sum() * 100
    tab["PUNTEGGIO_CTX"] = sum(tab[p] * w for p, w in PESI_CTX.items())
    return tab

def _causa(r):
    if r["dd_52w"] > -DD_GIALLO:
        return ""
    c = []
    if pd.notna(r["var_eps_12m"]) and r["var_eps_12m"] <= -0.15 or (pd.notna(r["trend_ebit"]) and r["trend_ebit"] <= SOGLIA_TREND_EBIT):
        c.append("utili")
    if pd.notna(r["trend_fcf"]) and r["trend_fcf"] <= SOGLIA_TREND_FCF:
        c.append("cassa (FCF)")
    if pd.notna(r["var_pe_12m"]) and r["var_pe_12m"] <= -0.20:
        c.append("multiplo")
    return " + ".join(c) if c else "altro: da verificare"

def bandiere_e_quadrante(tab):
    t = tab.copy()
    fr = ((t["trend_ebit"] <= SOGLIA_TREND_EBIT) | (t["trend_fcf"] <= SOGLIA_TREND_FCF)).values
    t["bandiera_fond"] = np.where(fr, "ROSSA", "VERDE")
    t["causa_calo"] = t.apply(_causa, axis=1)
    ammesso = (t["esclusione"] == "")
    soglia = t.loc[ammesso, "PUNTEGGIO"].quantile(QUALITA_PERCENTILE)
    alta = ammesso & (t["PUNTEGGIO"] >= soglia)
    incompleti = t["fcf_att"].isna() | t["fcf_pos"].isna() | t["fcf_ultimo_nd"]    # senza FCF recente non si può giudicare la qualità
    rosso = (t["bandiera_prezzo"] == "ROSSA") | (t["bandiera_fond"] == "ROSSA")
    cautela = t["bandiera_prezzo"] == "GIALLA"
    t["quadrante"] = np.select([~alta, alta & incompleti, alta & rosso, alta & cautela],
                               ["Fuori (qualità)", "Dati incompleti", "Da osservare", "Candidato con cautela"], default="Candidato")
    return t

def limita_per_sottosettore(df, n=MAX_PER_SOTTOSETTORE):
    """df già ordinato per qualità decrescente: tiene al massimo n titoli per sottosettore GICS."""
    return df.groupby("sottosettore", group_keys=False).head(n)

COL_PCT = ["dd_52w", "dd_3y", "dist_sma200", "mom_12_1", "vol_1y", "maxdd_5y", "trend_ebit", "trend_fcf",
           "fcf_yield", "pe_rel", "var_prezzo_12m", "var_eps_12m", "var_pe_12m"]
def vista(df, colonne):
    v = df[colonne].copy()
    for c in COL_PCT:
        if c in v:
            v[c] = (v[c] * 100).round(0)       # percentuali espresse in punti
    return v.round(1)

## 6. Correlazione, storico dei segnali e strumenti di diagnosi

In [ ]:
def _enb(r):
    c = r.corr().fillna(0)
    ev = np.clip(np.linalg.eigvalsh(c.values), 0, None)
    return ev.sum() ** 2 / (ev ** 2).sum()           # participation ratio degli autovalori

def _rendimenti():
    return PREZZI.pct_change().dropna(how="all").tail(252).dropna(axis=1, thresh=240).fillna(0)

def correlazioni(tickers, soglia=None):
    """Numero effettivo di titoli indipendenti e coppie molto correlate (rendimenti giornalieri, ultimo anno)."""
    soglia = CORR_ALTA if soglia is None else soglia
    r = _rendimenti()
    r = r[[t.replace(".", "-") for t in tickers if t.replace(".", "-") in r.columns]]
    c = r.corr()
    coppie = [(a, b, c.loc[a, b]) for i, a in enumerate(c.columns) for b in c.columns[i + 1:] if c.loc[a, b] >= soglia]
    coppie = pd.DataFrame(coppie, columns=["titolo_1", "titolo_2", "correlazione"]).sort_values("correlazione", ascending=False)
    return _enb(r), len(c), coppie

def enb_riferimento(n=30, prove=200, seed=0):
    """Termine di confronto: numero effettivo di titoli indipendenti in campioni casuali di n titoli dell'S&P 500."""
    rng = np.random.default_rng(seed)
    r = _rendimenti()
    cols = list(r.columns)
    valori = [_enb(r[list(rng.choice(cols, n, replace=False))]) for _ in range(prove)]
    return np.percentile(valori, [10, 50, 90])

def storico_segnali(ticker, mesi=12):
    """Mese per mese (solo prezzi): bandiera con persistenza e, a confronto, quella grezza."""
    df = serie_bandiere(PREZZI[ticker.replace(".", "-")].dropna())
    m = df.groupby(df.index.to_period("M")).last().tail(mesi)
    for c in ("dd_52w", "vs_sma200", "pend_sma200"):
        m[c] = (m[c] * 100).round(0)
    m["prezzo"] = m["prezzo"].round(2)
    return m[["prezzo", "dd_52w", "vs_sma200", "pend_sma200", "bandiera", "grezza"]]

def cerca_tag(ticker, testo, max_tag=25):
    """Diagnosi: elenca i tag us-gaap del titolo che contengono 'testo' (es. cerca_tag("TPL", "Acquire"))."""
    cik = uni.loc[uni["ticker"] == ticker, "cik"].iloc[0]
    facts = SESSION.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", timeout=30).json()["facts"].get("us-gaap", {})
    out = []
    for tag, node in facts.items():
        if testo.lower() in tag.lower():
            rows = node.get("units", {}).get("USD", [])
            if rows:
                u = max(rows, key=lambda r: r["end"])
                out.append((tag, u["end"], u["val"], len(rows)))
    return pd.DataFrame(out, columns=["tag", "ultimo_fine", "ultimo_valore", "n_valori"]).sort_values("ultimo_fine", ascending=False).head(max_tag)

## 7. Esecuzione: bilanci dell'intero S&P 500

In [ ]:
# ESECUZIONE
uni = carica_universo()
uni = uni[~uni["settore"].isin(ESCLUDI_SETTORI)].reset_index(drop=True)
print(f"{len(uni)} aziende da analizzare (esclusi: {', '.join(sorted(ESCLUDI_SETTORI))})")

righe, errori, DATI, TTM = [], [], {}, {}
for i, r in enumerate(uni.itertuples(), 1):
    try:
        df, ttm = scarica(r.cik, r.ticker)
        m = metriche(df, ttm) if df is not None else None
    except Exception as e:
        m = None
        errori.append((r.ticker, str(e)[:80]))
    if m:
        righe.append({"ticker": r.ticker, "nome": r.nome, "settore": r.settore, "sottosettore": r.sottosettore, **m})
        DATI[r.ticker], TTM[r.ticker] = df, ttm
    else:
        errori.append((r.ticker, "dati insufficienti"))
    if i % 50 == 0:
        print(f"  {i}/{len(uni)}")

tab = calcola_punteggi(pd.DataFrame(righe))
print(f"Analizzate: {len(tab)} | senza dati utilizzabili: {len(errori)} | ammesse dai filtri di qualità: {(tab['esclusione'] == '').sum()}")
motivi = tab["esclusione"].str.split("; ").explode()
print("\nMotivi di esclusione (un titolo può averne più di uno):")
print(motivi[motivi != ""].value_counts().to_string())
print("\nFonte dei capex:", tab["capex_fonte"].str.split(":").str[0].value_counts().to_dict())

394 aziende da analizzare (esclusi: Financials, Real Estate)
  50/394
  100/394
  150/394
  200/394
  250/394
  300/394
  350/394
Analizzate: 387 | senza dati utilizzabili: 7 | ammesse dai filtri di qualità: 303

Motivi di esclusione (un titolo può averne più di uno):
esclusione
FCF spesso negativo             58
perdite frequenti               21
copertura interessi bassa       21
debito netto/EBITDA alto        20
risultato operativo negativo    16

Fonte dei capex: {'standard': 368, 'alt': 16, 'mancante': 3}


## 8. Esecuzione: prezzi, contesto e quadrante

In [ ]:
# ESECUZIONE
controllo = [t for t in CASI_CONTROLLO + CASI_SANI if t in set(tab["ticker"])]
elenco = list(dict.fromkeys(list(tab.loc[tab["esclusione"] == "", "ticker"]) + controllo))
PREZZI = scarica_prezzi([t.replace(".", "-") for t in elenco])
print(f"Prezzi scaricati per {PREZZI.shape[1]} titoli, fino al {PREZZI.index[-1].date()}")

righe = []
for t in elenco:
    ty = t.replace(".", "-")
    if ty not in PREZZI.columns:
        continue
    mp = metriche_prezzo(PREZZI[ty])
    if mp is None:
        continue
    tt = TTM[t]
    eps, pe, var_eps, var_pe = scomposizione_12m(tt, mp)
    az = tt.get("azioni_ult", np.nan)
    mcap = mp["prezzo"] * az if np.isfinite(az) else np.nan
    fcf_att = tab.loc[tab["ticker"] == t, "fcf_att"].iloc[0]
    pe_st = pe_storia(DATI[t], PREZZI[ty])
    righe.append({"ticker": t, **mp,
                  "fcf_yield": fcf_att / mcap if np.isfinite(mcap) and mcap > 0 and pd.notna(fcf_att) else np.nan,
                  "pe_ttm": pe, "pe_storia": pe_st,
                  "pe_rel": pe / pe_st - 1 if np.isfinite(pe) and np.isfinite(pe_st) else np.nan,
                  "var_eps_12m": var_eps, "var_pe_12m": var_pe})

tabc = tab.merge(pd.DataFrame(righe), on="ticker", how="inner")
tabc = score_contesto(tabc)
tabc = bandiere_e_quadrante(tabc).sort_values("PUNTEGGIO", ascending=False).reset_index(drop=True)
print(tabc["quadrante"].value_counts().to_string())
copertura = tabc[["pe_rel", "fcf_yield", "trend_ebit", "trend_fcf", "var_eps_12m", "var_pe_12m", "mom_12_1", "fcf_conv", "fcf_min_margin", "sh_cagr"]].isna().mean().mul(100).round(0)
print("\nDati mancanti per indicatore (% dei titoli):")
print(copertura.to_string())

Prezzi scaricati per 305 titoli, fino al 2026-11-23
quadrante
Fuori (qualità)          214
Candidato                 56
Da osservare              17
Candidato con cautela     15
Dati incompleti            3

Dati mancanti per indicatore (% dei titoli):
pe_rel             3.0
fcf_yield          3.0
trend_ebit        16.0
trend_fcf          3.0
var_eps_12m        2.0
var_pe_12m         3.0
mom_12_1           0.0
fcf_conv           1.0
fcf_min_margin     1.0
sh_cagr            1.0


In [ ]:
colonne = ["ticker", "nome", "sottosettore", "PUNTEGGIO", "PUNTEGGIO_CTX", "dd_52w", "dist_sma200",
           "mom_12_1", "trend_ebit", "trend_fcf", "fcf_yield", "pe_rel", "bandiera_prezzo", "bandiera_fond"]
cand = limita_per_sottosettore(tabc[tabc["quadrante"].isin(["Candidato", "Candidato con cautela"])])
print("CANDIDATI: qualità alta, nessuna bandiera rossa (massimo", MAX_PER_SOTTOSETTORE, "per sottosettore). 'con cautela' = bandiera di prezzo gialla.")
print("Percentuali espresse in punti.")
vista(cand, ["quadrante"] + colonne).head(40)

CANDIDATI: qualità alta, nessuna bandiera rossa (massimo 2 per sottosettore). 'con cautela' = bandiera di prezzo gialla.
Percentuali espresse in punti.


,quadrante,ticker,nome,sottosettore,PUNTEGGIO,PUNTEGGIO_CTX,dd_52w,dist_sma200,mom_12_1,trend_ebit,trend_fcf,fcf_yield,pe_rel,bandiera_prezzo,bandiera_fond
3,Candidato,CF,CF Industries,Fertilizers & Agricultural Chemicals,76.0,60.0,-16.0,-1.0,57.0,44.0,-8.0,12.0,-11.0,VERDE,VERDE
5,Candidato,SNA,Snap-on,Industrial Machinery & Supplies & Components,73.8,58.3,-13.0,-4.0,11.0,-1.0,7.0,6.0,31.0,VERDE,VERDE
6,Candidato,CDNS,Cadence Design Systems,Application Software,72.4,38.4,-27.0,-7.0,5.0,19.0,11.0,2.0,-19.0,VERDE,VERDE
7,Candidato,FAST,Fastenal,Trading Companies & Distributors,72.1,56.5,-6.0,10.0,35.0,13.0,12.0,2.0,40.0,VERDE,VERDE
8,Candidato,PM,Philip Morris International,Tobacco,71.7,56.6,-14.0,1.0,33.0,12.0,27.0,5.0,59.0,VERDE,VERDE
9,Candidato,VRSN,Verisign,Internet Services & Infrastructure,71.3,59.5,-1.0,14.0,22.0,4.0,-4.0,4.0,23.0,VERDE,VERDE
10,Candidato,MNST,Monster Beverage,Soft Drinks & Non-alcoholic Beverages,71.1,46.3,-15.0,-2.0,28.0,13.0,-2.0,2.0,107.0,VERDE,VERDE
11,Candidato,YUM,Yum! Brands,Restaurants,70.7,70.4,-1.0,8.0,2.0,69.0,-1.0,4.0,-47.0,VERDE,VERDE
13,Candidato con cautela,FTNT,Fortinet,Systems Software,70.3,46.1,-33.0,-1.0,131.0,25.0,54.0,4.0,2.0,GIALLA,VERDE
14,Candidato con cautela,ADBE,Adobe Inc.,Application Software,69.9,44.0,-37.0,-9.0,-16.0,6.0,8.0,12.0,-70.0,GIALLA,VERDE


In [ ]:
for t in ["APP", "RL", "ABNB"]:
    for parola in ("PropertyPlant", "Software", "Capital"):
        print(t, parola); print(cerca_tag(t, parola, 6)[["tag", "ultimo_fine"]].to_string(index=False))

APP PropertyPlant
                                                                                                tag ultimo_fine
 PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization  2026-09-30
      PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAccumulatedDepreciationAndAmortization  2025-12-31
PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetBeforeAccumulatedDepreciationAndAmortization  2025-12-31
                                                         PaymentsToAcquirePropertyPlantAndEquipment  2024-12-31
                                                    DeferredTaxLiabilitiesPropertyPlantAndEquipment  2024-12-31
                                                                       PropertyPlantAndEquipmentNet  2022-09-30
APP Software
                                 tag ultimo_fine
           PaymentsToDevelopSoftware  2023-09-30
CapitalizedComputerSoftwareAdditions  2022-12-31
APP Capital
                          

In [ ]:
oss = tabc[tabc["quadrante"] == "Da osservare"]
print("DA OSSERVARE: qualità alta ma con bandiera rossa di prezzo o di fondamentali.")
vista(oss, colonne + ["causa_calo"]).head(30)

DA OSSERVARE: qualità alta ma con bandiera rossa di prezzo o di fondamentali.


,ticker,nome,sottosettore,PUNTEGGIO,PUNTEGGIO_CTX,dd_52w,dist_sma200,mom_12_1,trend_ebit,trend_fcf,fcf_yield,pe_rel,bandiera_prezzo,bandiera_fond,causa_calo
1,GOOG,Alphabet Inc. (Class C),Interactive Media & Services,78.1,38.0,-30.0,-16.0,10.0,22.0,-73.0,1.0,-39.0,GIALLA,ROSSA,cassa (FCF) + multiplo
2,TPL,Texas Pacific Land Corporation,Oil & Gas Exploration & Production,76.2,22.1,-52.0,-35.0,16.0,23.0,-24.0,1.0,5.0,ROSSA,VERDE,multiplo
4,META,Meta Platforms,Interactive Media & Services,75.8,31.3,-23.0,-5.0,9.0,-5.0,-38.0,2.0,-16.0,GIALLA,ROSSA,cassa (FCF) + multiplo
12,AMAT,Applied Materials,Semiconductor Materials & Equipment,70.6,21.4,-52.0,-22.0,72.0,10.0,-1.0,2.0,61.0,ROSSA,VERDE,altro: da verificare
34,ROL,"Rollins, Inc.",Environmental & Facilities Services,66.5,35.7,-41.0,-14.0,-41.0,1.0,3.0,4.0,-25.0,ROSSA,VERDE,multiplo
40,AMCR,Amcor,Paper & Plastic Packaging Products & Materials,65.5,59.2,-11.0,3.0,17.0,-0.0,-37.0,4.0,-30.0,VERDE,ROSSA,
41,AKAM,Akamai Technologies,Internet Services & Infrastructure,65.5,25.1,-45.0,-20.0,7.0,-37.0,-47.0,4.0,19.0,ROSSA,ROSSA,utili + cassa (FCF)
43,LRCX,Lam Research,Semiconductor Materials & Equipment,65.2,24.2,-52.0,-26.0,66.0,18.0,34.0,2.0,34.0,ROSSA,VERDE,altro: da verificare
51,INTU,Intuit,Application Software,64.3,37.5,-59.0,-20.0,-45.0,6.0,4.0,12.0,-74.0,ROSSA,VERDE,multiplo
53,WMB,Williams Companies,Oil & Gas Storage & Transportation,64.1,43.1,-16.0,-9.0,25.0,11.0,-111.0,-0.0,53.0,VERDE,ROSSA,


In [ ]:
inc = tabc[tabc["quadrante"] == "Dati incompleti"]
print("DATI INCOMPLETI: qualità alta ma FCF non calcolabile (capex mancanti). Da verificare a mano, ad esempio con cerca_tag(ticker, 'Acquire').")
vista(inc, ["ticker", "nome", "PUNTEGGIO", "capex_fonte", "capex_mancanti", "dd_52w", "bandiera_prezzo"])

DATI INCOMPLETI: qualità alta ma FCF non calcolabile (capex mancanti). Da verificare a mano, ad esempio con cerca_tag(ticker, 'Acquire').


,ticker,nome,PUNTEGGIO,capex_fonte,capex_mancanti,dd_52w,bandiera_prezzo
22,APP,AppLovin,68.3,standard,1,-49.0,ROSSA
66,RL,Ralph Lauren Corporation,62.3,standard,1,-6.0,VERDE
68,ABNB,Airbnb,61.7,standard,3,-9.0,VERDE


## 9. Quanti titoli indipendenti ci sono davvero?

In [ ]:
top = list(pd.concat([cand, oss]).sort_values("PUNTEGGIO", ascending=False)["ticker"].head(30))
enb, n, coppie = correlazioni(top)
p10, p50, p90 = enb_riferimento(n=n)
print(f"Titoli analizzati: {n} | numero effettivo di titoli indipendenti: {enb:.1f}")
print(f"Confronto: in campioni casuali di {n} titoli dell'S&P 500 la mediana è {p50:.1f} (10°-90° percentile: {p10:.1f}-{p90:.1f})")
print(f"Coppie con correlazione >= {CORR_ALTA:.0%} (rendimenti giornalieri, ultimo anno):")
coppie.round(2).head(25)

Titoli analizzati: 30 | numero effettivo di titoli indipendenti: 13.6
Confronto: in campioni casuali di 30 titoli dell'S&P 500 la mediana è 11.5 (10°-90° percentile: 9.9-13.3)
Coppie con correlazione >= 70% (rendimenti giornalieri, ultimo anno):


,titolo_1,titolo_2,correlazione


## 10. Casi di controllo

In [ ]:
casi = tabc[tabc["ticker"].isin(controllo)]
mancanti = [t for t in CASI_CONTROLLO + CASI_SANI if t not in set(casi["ticker"])]
if mancanti:
    print("Non disponibili (esclusi per settore, senza dati o senza prezzi):", mancanti)
vista(casi, ["ticker", "quadrante", "PUNTEGGIO", "esclusione", "PUNTEGGIO_CTX", "dd_52w", "dist_sma200", "trend_ebit", "trend_fcf",
             "var_prezzo_12m", "var_eps_12m", "var_pe_12m", "fcf_yield", "bandiera_prezzo", "bandiera_fond", "causa_calo", "capex_fonte"])

,ticker,quadrante,PUNTEGGIO,esclusione,PUNTEGGIO_CTX,dd_52w,dist_sma200,trend_ebit,trend_fcf,var_prezzo_12m,var_eps_12m,var_pe_12m,fcf_yield,bandiera_prezzo,bandiera_fond,causa_calo,capex_fonte
0,NRG,Fuori (qualità),79.3,FCF spesso negativo,14.3,-46.0,-25.0,33.0,-10.0,-38.0,-16.0,-26.0,3.0,ROSSA,VERDE,utili + multiplo,standard
1,GOOG,Da osservare,78.1,,38.0,-30.0,-16.0,22.0,-73.0,-3.0,88.0,-49.0,1.0,GIALLA,ROSSA,cassa (FCF) + multiplo,standard
2,TPL,Da osservare,76.2,,22.1,-52.0,-35.0,23.0,-24.0,-12.0,23.0,-29.0,1.0,ROSSA,VERDE,multiplo,alt:PaymentsToAcquireRoyaltyInterestsInMiningP...
6,CDNS,Candidato,72.4,,38.4,-27.0,-7.0,19.0,11.0,1.0,29.0,-22.0,2.0,VERDE,VERDE,multiplo,standard
15,AOS,Candidato,69.5,,58.0,-18.0,5.0,-9.0,3.0,5.0,-3.0,9.0,6.0,VERDE,VERDE,,standard
24,VST,Fuori (qualità),68.0,FCF spesso negativo,36.7,-31.0,-18.0,122.0,98.0,-28.0,140.0,-70.0,6.0,GIALLA,VERDE,multiplo,standard


In [ ]:
# Bandiere di prezzo mese per mese, con persistenza ("bandiera") e senza ("grezza"). Esempio: storico_segnali("NRG")
storico_segnali("NRG")

,prezzo,dd_52w,vs_sma200,pend_sma200,bandiera,grezza
Date,,,,,,
2025-12,156.92,-11.0,8.0,5.0,VERDE,VERDE
2026-01,150.41,-14.0,-1.0,4.0,VERDE,VERDE
2026-02,176.90,-3.0,13.0,4.0,VERDE,VERDE
2026-03,144.46,-21.0,-8.0,-0.0,VERDE,GIALLA
2026-04,153.79,-15.0,-2.0,0.0,VERDE,VERDE
2026-05,132.95,-27.0,-15.0,-1.0,GIALLA,GIALLA
2026-06,144.82,-20.0,-6.0,-1.0,GIALLA,GIALLA
2026-07,133.15,-27.0,-12.0,-2.0,GIALLA,GIALLA
2026-08,109.58,-40.0,-25.0,-3.0,GIALLA,ROSSA


In [ ]:
tabc.to_csv("classifica_v22_qualita_contesto.csv", index=False)
# In Colab:  from google.colab import files; files.download("classifica_v22_qualita_contesto.csv")